In [0]:
RAW_PATH = "/Volumes/workspace/default/raw_labs"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_labs

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.swish_orders;
CREATE TABLE IF NOT EXISTS workspace.default.swish_customers;

In [0]:
RAW_ORDER_TBL = "workspace.default.swish_orders"
RAW_CUSTOMER_TBL = "workspace.default.swish_customers"

### 1.1 Creating and writing Delta tables

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/swish_orders.csv")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("store").saveAsTable(BRONZE_ORDER_TBL)

orders_df = spark.table(BRONZE_ORDER_TBL)
display(orders_df.limit(5))

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/swish_customers.csv")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE_CUSTOMER_TBL)

customers_df = spark.table(BRONZE_CUSTOMER_TBL)
display(customers_df.limit(5))

### 1.2 Time travel

In [0]:
%sql
-- Query a prior table
SELECT * FROM workspace.default.swish_orders VERSION AS OF 1;

-- Rollback to an old version
RESTORE TABLE workspace.default.swish_orders TO VERSION AS OF 1;

-- Inspect the full history
DESCRIBE HISTORY workspace.default.swish_orders;

### 1.3 Schema Enforcement

In [0]:
df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(BRONZE_ORDER_TBL)

### 1.4 Upsert with MERGE INTO

In [0]:
%sql
MERGE INTO workspace.default.swish_orders AS target
USING workspace.default.swish_orders_update AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

### 1.5 Optimize & Z-Order & Liquid Clustering

In [0]:
%sql
-- Optmize -> Compacts small files into larger ones
OPTIMIZE workspace.default.swish_orders;

-- Z-Order -> Store related data near each other so queries scan less data and run faster.
OPTIMIZE workspace.default.swish_orders ZORDER BY (city, store);

-- Liquid Clustering -> Instead of manually deciding the partition column, let Databricks choose the best one for you.
ALTER TABLE workspace.default.swish_orders CLUSTER BY store;

### 1.6 Vaccum

In [0]:
%sql
-- Removes files that are no longer referenced by table and older than the retention window
VACCUM workspace.default.swish_orders RETAIN 168 HOURS; -- 168 hours = 7 days

## 2. Medallion Architecture

In [0]:
%sql
SELECT * FROM workspace.default.swish_customers
LIMIT (5);

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, month, to_date, col, when, year, sum, avg, lower

bronze_orders_df = spark.table(RAW_ORDER_TBL) \
    .withColumn("_ingest_time", current_timestamp())

bronze_customer_df = spark.table(RAW_CUSTOMER_TBL) \
    .withColumn("_ingest_time", current_timestamp())

bronze_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_orders")
bronze_customer_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_customers")

silver_orders_df = spark.table("bronze_orders") \
    .dropDuplicates(["order_id"]) \
    .filter(col("order_id").isNotNull()) \
    .withColumn("order_month", month("order_timestamp")) \
    .withColumn("order_year", year("order_timestamp")) \
    .withColumn("order_date", to_date("order_timestamp")) \
    .withColumn("status_indicator", when(lower(col("order_status")) == "delivered", 1).otherwise(0))

silver_customers_df = spark.table("bronze_customers") \
    .dropDuplicates(["customer_id"]) \
    .withColumn("customer_segment_indicator", when(col("customer_segment") == "Premium", 1).otherwise(0)) \
    .withColumn("account_status_indicator", when(col("account_status") == "Active", 1).otherwise(0)) \
    .withColumnRenamed("total_orders_before_aug", "total_prev_orders") \
    .select(
        "customer_id",
        "signup_date",
        "neighbourhood",
        "customer_segment_indicator",
        "account_status_indicator",
        "total_prev_orders",
        "age",
        "account_status",
        "customer_segment"
    )

silver_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_orders")
silver_customers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_customers")

silver_order = spark.table("silver_orders")
silver_customer = spark.table("silver_customers")

silver_swish_enriched = silver_order.join(silver_customer, "customer_id", "inner")

silver_swish_enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_swish_enriched")

gold_swish = spark.table("silver_swish_enriched") \
    .groupBy("order_month", "order_year", "customer_segment", "account_status") \
    .agg(
        sum("status_indicator").alias("total_orders"),
        sum("order_value_inr").alias("total_value"),
        avg("delivery_time_min").alias("avg_delivery_time")
    )

gold_swish.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_swish")

## 3. Ingestion Pattern

### 3.1 Auto Loader (cloudFiles)

In [0]:
# Auto Loader incrementally and efficiently ingests new files as they land in cloud storage, without you having to list an entire directory on every run. It tracks which files have already been processed using scalable checkpointed state (RocksDB-backed), and can also auto-detect and evolve schema.

DEMO_DB_PATH = "workspace.default.demopath"
DEMO_WRITE_PATH = "workspace.default.demowrite"

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", f"{DEMO_DB_PATH}/schema") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.maxFilesPerTrigger", 1) \
    .load(f"{DEMO_DB_PATH}/demo_data")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{DEMO_DB_PATH}/checkpoint") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(f"{DEMO_WRITE_PATH}")

### 3.2 COPY INTO

In [0]:
%sql
CREATE TABLE demowrite;

In [0]:
%sql
-- COPY INTO is a simpler, SQL-native, idempotent, re-runnable bulk-load command — good for smaller numbers of files or ad-hoc/one-off loads where full Auto Loader streaming infrastructure is overkill.

COPY INTO workspace.default.demowrite
FROM '/Volumes/workspace/default/raw_labs/demo.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

## 4. Structured Streaming

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS raw_stream;

In [0]:
RAW_STREAM_PATH = "/Volumes/workspace/default/raw_stream"

### 4.1 Triggers

#### Read from the source

In [0]:
silver_orders_path = spark.table("silver_orders")
stream_df = spark.readStream.format("delta").load(silver_orders_path)

#### 4.1.1 processingTime

In [0]:
# outputMode -> append, update, complete
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(processingTime="1 minute") \
    .start(f"{RAW_STREAM_PATH}t")

#### 4.1.2 availableNow

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start(f"{RAW_STREAM_PATH}")

#### 4.1.3 Checkpoint

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{RAW_STREAM_PATH}/_checkpoints") \
    .trigger(availableNow=True) \
    .start(f"{RAW_STREAM_PATH}")

### 4.2 Watermarking

In [0]:
# Watermarking tells Spark to keep late data up to this threshold, then discard older state

from pyspark.sql.functions import col, window, count
stream_df = stream_df.withWatermark("order_timestamp", "10 minutes") \
                    .groupBy(window(col("order_timestamp"), "5 minutes"), col("city")) \
                    .agg(count("*").alias("total_orders"))

### 4.3 Stream Static Join

In [0]:
orders_stream = spark.readStream.table("silver_orders")
customers = spark.table("silver_customers")

enriched = orders_stream.join(customers, "customer_id", "left")

### 4.4 Stream Stream Join

In [0]:
customer_demo = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/customer_demo.csv")
customer_demo.write.format("delta").mode("overwrite").saveAsTable("customer_demo")
order_demo = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/order_demo.csv")
order_demo.write.format("delta").mode("overwrite").saveAsTable("order_demo")

display(customer_demo.limit(2))
display(order_demo.limit(2))

In [0]:
from pyspark.sql.functions import col, expr

order_stream = (
    spark.readStream.table("order_demo")
    .withWatermark("order_timestamp", "1 hour")
)

customer_stream = (
    spark.readStream.table("customer_demo")
    .withWatermark("customer_timestamp", "1 hour")
)

join_condt = """
o.customer_id = c.customer_id
AND c.customer_timestamp >= o.order_timestamp
AND c.customer_timestamp < o.order_timestamp + INTERVAL 5 HOURS
"""

joined_stream = (
    order_stream.alias("o")
    .join(customer_stream.alias("c"), expr(join_condt), "left")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("o.amount"),
        col("o.order_timestamp"),
        col("c.customer_name"),
        col("c.city"),
        col("c.customer_timestamp")
    )
)
display(joined_stream.limit(2))
query = (
    joined_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/raw_stream/_checkpoints-joined")
    .trigger(availableNow=True)
    .start("/Volumes/workspace/default/raw_stream/joined")
)

## 5. Delta Live Tables (DLT)

### 5.1 Declaring a pipeline

In [0]:
import dlt
from pyspark.sql.functions import col

@dlt.table(comment="Raw events landed into Auto Loader")
def bronze_orders():
    return (spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").load(f"{RAW_PATH}/order_demo.csv"))

@dlt.table(comment="Cleaned, deduplicated order_id")
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("valid_amount", "amount > 0")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .dropDuplicates(["order_id"])
    )
    
@dlt.table(comment="Hourly aggregate for dashboard")
def gold_hourly_counts():
    return (dlt.read("silver_orders").groupBy("city").count())

'''
Data Quality
@dlt.expect(name, condition) --> Row is kept; failure is logged
@dlt.expect_or_drop(name, condition) --> Failing rows are silently dropped from the output
@dlt.expect_or_fail(name, condition) --> Pipeline run fails immediately if any row violates condition
'''

## 6. Unity Catalog & Governance

### 6.1 Three level namespace

In [0]:
%sql
-- Correct usage to query a table:
SELECT * FROM workspace.default.silver_orders;

-- Use commands to set catalog and schema, then query table:
USE CATALOG workspace;
USE SCHEMA default;
SELECT * FROM silver_orders;

### 6.2 Access Control

In [0]:
%sql
GRANT SELECT ON TABLE workspace.default.bronze_orders TO 'data-analyst';
GRANT MODIFY, SELECT ON SCHEMA workspace.default TO 'data-engineers';
REVOKE SELECT ON TABLE workspace.select.swish_orders FROM 'contractor-group';

### 6.3 Row Level Security

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.gold_swish AS
SELECT
  order_month,
  order_year,
  customer_segment,
  account_status,
  total_orders,
  avg_delivery_time,
  CASE
    WHEN is_member('finance_team') THEN total_value
    ELSE NULL
  END AS total_value
FROM workspace.default.gold_swish;

## 7. Performance Tuning & Optmization

### 7.1 Partitioning

In [0]:
DEMO_PATH = "/Volumes/workspace/default/raw_labs"

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{DEMO_PATH}/demo.csv")

df.write.format("delta").mode("overwrite").partitionBy("city").saveAsTable("demo")
partitions = df.rdd.getNumPartitions()
print(partitions)

### Broadcast Join

In [0]:
from pyspark.sql.functions import broadcast
silver_order = spark.table("silver_orders")
silver_customer = spark.table("silver_customers")

silver_swish_enriched = silver_order.join(
    broadcast(silver_customer),
    "customer_id",
    "inner"
)
silver_swish_enriched.write.format("delta").mode("overwrite")

### 7.2 Caching

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{DEMO_PATH}/demo.csv")
df.cache()
df.count()
spark.sql("CACHE SELECT * FROM workspace.default.demo")